<a href="https://colab.research.google.com/github/canariocfet/Nexus/blob/Modelagem_DataSet_Enriquecido/Aprendizado_n_supervisonado_22_09_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd
import io
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
# CORREÇÃO DA IMPORTAÇÃO AQUI:
from sklearn.metrics.cluster import contingency_matrix
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeClassifier, export_text
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics.cluster import contingency_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.cluster import contingency_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA


print("Clique no botão abaixo para selecionar o arquivo csv do seu computador:")
# 1. Abre a janela para você escolher o arquivo
uploaded = files.upload()

# 2. Pega o nome do arquivo que você enviou automaticamente
nome_do_arquivo = list(uploaded.keys())[0]
print(f"\nArquivo '{nome_do_arquivo}' carregado com sucesso!")

# 3. Lê o arquivo carregado diretamente para a tabela do Pandas
df = pd.read_csv(io.BytesIO(uploaded[nome_do_arquivo]))

# 4. Mostra as primeiras 5 linhas para confirmar
df.head()


Clique no botão abaixo para selecionar o arquivo csv do seu computador:


In [ ]:
df.columns

### 📝 Dicionário de Variáveis (Dataset)

A tabela abaixo apresenta a tradução e a descrição de cada métrica textual presente nas colunas originais do conjunto de dados:

| Coluna Original (Inglês) | Coluna Traduzida | Descrição / Significado |
| :--- | :--- | :--- |
| `author` | `autor` | Autor do texto ou da notícia. |
| `link` | `link` | URL ou ligação da publicação original. |
| `category` | `categoria` | Categoria, tema ou editoria do texto. |
| `date_of_publication` | `data_publicacao` | Data em que o conteúdo foi publicado. |
| `number_of_tokens` | `total_tokens` | Contagem total de tokens (palavras, números e pontuações). |
| `number_of_words_without_punctuation` | `total_palavras_sem_pontuacao` | Total de palavras no texto, ignorando os sinais de pontuação. |
| `number_of_types` | `total_tipos_palavras` | Quantidade de palavras únicas (tamanho do vocabulário). |
| `number_of_links_inside_news` | `total_links_na_noticia` | Quantidade de hiperlinks inseridos dentro do corpo do texto. |
| `number_of_words_in_upper_case` | `total_palavras_maiusculas` | Contagem de palavras escritas inteiramente em letras maiúsculas. |
| `number_of_verbs` | `total_verbos` | Quantidade total de verbos identificados. |
| `number_of_subjunctive_and_imperative_verbs` | `total_verbos_subjuntivo_imperativo` | Verbos que expressam desejo, incerteza (subjuntivo) ou ordem/pedido (imperativo). |
| `number_of_nouns` | `total_substantivos` | Quantidade total de substantivos. |
| `number_of_adjectives` | `total_adjetivos` | Quantidade total de adjetivos (características/qualidades). |
| `number_of_adverbs` | `total_adverbios` | Quantidade total de advérbios (modificadores de tempo, modo, etc.). |
| `number_of_modal_verbs` | `total_verbos_modais` | Verbos que indicam modalidade, como capacidade ou obrigação (ex: dever, poder). |
| `number_of_singular_first_and_second_personal_pronouns` | `total_pronomes_singular_1a_2a` | Pronomes pessoais de 1ª e 2ª pessoa do singular (ex: eu, tu, você). |
| `number_of_plural_first_personal_pronouns` | `total_pronomes_plural_1a` | Pronomes pessoais de 1ª pessoa do plural (ex: nós). |
| `number_of_pronouns` | `total_pronomes` | Contagem geral de todos os tipos de pronomes no texto. |
| `pausality` | `pausalidade` | Índice que mede a frequência de pausas na leitura (baseado na pontuação). |
| `number_of_characters` | `total_caracteres` | Contagem total de caracteres (letras, espaços e símbolos). |
| `average_sentence_length` | `tamanho_medio_frase` | Média de palavras ou caracteres por frase do texto. |
| `average_word_length` | `tamanho_medio_palavra` | Média de caracteres por palavra. |
| `emotiveness` | `emotividade` | Proporção de adjetivos e advérbios em relação a substantivos e verbos. |
| `diversity` | `diversidade_lexical` | Grau de variedade do vocabulário (proporção entre palavras únicas e totais). |
| `label` | `rotulo` | Classificação final do texto (ex: se é uma notícia Real ou Fake). |


In [ ]:
# 1. Definindo as features estilométricas do seu dataset
features_numericas = [
    'number_of_tokens', 'number_of_words_without_punctuation', 'number_of_types',
    'number_of_links_inside_news', 'number_of_words_in_upper_case', 'number_of_verbs',
    'number_of_subjunctive_and_imperative_verbs', 'number_of_nouns', 'number_of_adjectives',
    'number_of_adverbs', 'number_of_modal_verbs', 'number_of_singular_first_and_second_personal_pronouns',
    'number_of_plural_first_personal_pronouns', 'number_of_pronouns', 'pausality',
    'number_of_characters', 'average_sentence_length', 'average_word_length', 'emotiveness', 'diversity'
]

# 2. Limpeza de dados nulos
df_limpo = df.dropna(subset=features_numericas + ['label']).copy()
X = df_limpo[features_numericas]
y = df_limpo['label']

# 3. Padronização (Essencial para o K-Means)
scaler = StandardScaler()
X_normalizado = scaler.fit_transform(X)

# 4. Construindo o Modelo K-Means para 2 grupos
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
grupos_kmeans = kmeans.fit_predict(X_normalizado)

# 5. Avaliação do Alinhamento com a Realidade
matriz = contingency_matrix(y, grupos_kmeans)

print("="*50)
print(" MATRIZ DE COMPARAÇÃO - K-MEANS ")
print("="*50)
print(f"Classes Reais (Linhas): {list(y.unique())}")
print("Grupos do Modelo (Colunas): Cluster 0 | Cluster 1")
print(matriz)

# Cálculo de acurácia não supervisionada (Pureza do Cluster)
acerto_direto = np.diag(matriz).sum() / matriz.sum()
acerto_invertido = np.fliplr(matriz).diagonal().sum() / matriz.sum()
pureza = max(acerto_direto, acerto_invertido)

print(f"\nTaxa de Alinhamento (Pureza): {pureza:.2%}")


## Modelo 1: Agrupamento com K-Means

### 1. Explicação Teórica do Método
O **K-Means** é um algoritmo de aprendizado não supervisionado baseado em **distância geométrica** (geralmente a Distância Euclidiana). O objetivo principal do modelo é particionar um conjunto de dados em *K* grupos distintos (clusters). No nosso caso, definimos `n_clusters=2` para avaliar se o modelo consegue separar espontaneamente as notícias em "Falsas" e "Verdadeiras" utilizando métricas textuais.

O funcionamento do algoritmo segue quatro passos principais:
1. **Inicialização:** O algoritmo escolhe aleatoriamente 2 pontos no espaço matemático para servirem de "centroides" iniciais (os centros de cada grupo).
2. **Associação:** Cada notícia do dataset é associada ao centroide mais próximo, baseando-se nas suas características numéricas (recursos estilométricos como tamanho de frase, uso de pontuação, emotividade, etc.).
3. **Atualização:** O centroide é recalculado, movendo-se para a média matemática de todos os pontos atribuídos àquele grupo.
4. **Convergência:** Os passos 2 e 3 se repetem de forma iterativa até que os centroides parem de mudar de posição, indicando que os clusters estão consolidados.

Como o K-Means calcula distâncias matemáticas complexas no espaço multidimensional, a aplicação prévia do `StandardScaler` foi essencial para reescalar todas as variáveis para que possuam **média 0 e desvio padrão 1**, garantindo pesos iguais para todas as métricas gramaticais durante o agrupamento.

---

### 2. Análise Prática dos Resultados

O algoritmo foi executado sem ter acesso prévio às respostas (`label`) e gerou a seguinte matriz de contingência ao cruzar os grupos descobertos com a realidade:

```text
==================================================
 MATRIZ DE COMPARAÇÃO - K-MEANS
==================================================
Classes Reais (Linhas): ['fake', 'true']
Grupos do Modelo (Colunas): Cluster 0 | Cluster 1
[[8053   20]
 [1178 1333]]

Taxa de Alinhamento (Pureza): 88.68%
```

#### Interpretação Detalhada da Matriz:
* **Cluster 0 (O grupo das Fake News):** Este grupo concentrou a grande maioria das notícias falsas (**8.053 notícias**). No entanto, ele também puxou erroneamente **1.178** notícias verdadeiras para dentro dele.
* **Cluster 1 (O grupo das Notícias Verdadeiras):** Este grupo foi extremamente conservador e preciso. Ele isolou **1.333** notícias verdadeiras e cometeu pouquíssimos erros crassos, incluindo apenas **20** notícias fakes dentro dele.

#### Conclusão Estatística do Modelo:
O K-Means obteve uma **Taxa de Alinhamento (Pureza) de 88.68%**. Isso prova que as características estilométricas e estruturais das notícias falsas (como tamanho do texto, quantidade de palavras em caixa alta e uso de verbos) são fortemente marcantes a ponto de um algoritmo baseado puramente em distância geométrica separar as fakes das verdadeiras com alta taxa de sucesso de forma 100% autônoma.

O comportamento do modelo indica que as notícias falsas seguem um padrão estilístico muito rígido e homogêneo (agrupadas quase totalmente no Cluster 0), enquanto as notícias reais possuem maior variação em sua estrutura escrita, fazendo com que uma parte delas se misturasse estatisticamente com a distribuição das fakes.


In [ ]:
# ==========================================
# 1. MÉTODO DO COTOVELO (JUSTIFICATIVA DO K)
# ==========================================
inercia = []
K_range = range(1, 6) # Testando de 1 a 5 clusters

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_normalizado)
    inercia.append(km.inertia_)

# Plotando o gráfico do cotovelo
plt.figure(figsize=(8, 4))
plt.plot(K_range, inercia, 'bx-')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia (Soma dos Quadrados das Distâncias)')
plt.title('Método do Cotovelo para Escolha do K Ideal')
plt.grid(True)
plt.show()

# ==========================================
# 2. ANÁLISE COMPORTAMENTAL DAS FEATURES
# ==========================================
# Adiciona os clusters gerados de volta ao DataFrame original (sem normalizar)
df_limpo['Cluster_KMeans'] = grupos_kmeans

# Seleciona algumas features principais para comparar a média entre os clusters
features_analise = ['average_sentence_length', 'number_of_words_in_upper_case', 'emotiveness', 'diversity']

print("\n" + "="*50)
print(" MÉDIA DAS CARACTERÍSTICAS POR CLUSTER ")
print("="*50)
perfil_clusters = df_limpo.groupby('Cluster_KMeans')[features_analise].mean()
print(perfil_clusters)


### 3. Análise do Perfil de Estilometria dos Clusters (Resultados Reais)

Ao mapear as características originais das notícias que foram agrupadas pelo K-Means, identificamos assinaturas de escrita radicalmente distintas entre os dois grupos, o que explica como o modelo conseguiu separá-las tão bem sem supervisão:

| Métrica Avaliada | Cluster 0 (Fakes dominantes) | Cluster 1 (Reais dominantes) |
| :--- | :---: | :---: |
| **Média do tamanho das frases** (`average_sentence_length`) | **16.20** palavras | **21.57** palavras |
| **Palavras em Caixa Alta** (`number_of_words_in_upper_case`) | **2.72** por texto | **16.05** por texto |
| **Índice de Emotividade** (`emotiveness`) | **0.14** | **0.23** |
| **Diversidade Lexical** (`diversity`) | **0.79** | **0.43** |

#### Conclusões Linguísticas a partir das Médias:
1. **Sintaxe e Estrutura das Frases:** O Cluster 0 (Fakes) utiliza frases bem mais curtas (média de 16.20 palavras) do que o Cluster 1 (21.57 palavras). Isso demonstra que notícias falsas tendem a adotar uma escrita direta, simples e de fácil leitura para engajar o leitor rapidamente.
2. **Uso de Caixa Alta:** Curiosamente, o Cluster 1 apresentou uma quantidade substancialmente maior de palavras em caixa alta (16.05 contra 2.72). No contexto do jornalismo profissional brasileiro (*Fake.br-Corpus*), isso geralmente indica a presença rigorosa de siglas institucionais, nomes de entidades públicas e siglas de partidos/órgãos governamentais, reforçando a natureza formal e documental das notícias reais.
3. **Diversidade Lexical:** O Cluster 0 apresentou uma taxa de diversidade lexical surpreendentemente alta (0.79). Isso ocorre frequentemente em notícias falsas devido ao uso de termos adjetivos variados e dramáticos, ou textos de extensão total mais curta (onde a repetição de palavras é menor), em oposição a reportagens profundas e longas que tendem a repetir termos técnicos.


In [ ]:
# Calcula a média do Silhouette Score para K=2
score = silhouette_score(X_normalizado, grupos_kmeans)

print("="*50)
print(" TESTE DA SILHUETA - K-MEANS ")
print("="*50)
print(f"Coeficiente de Silhueta Geral: {score:.4f}")


### 4. Validação por Coeficiente de Silhueta (Silhouette Score)
O **Coeficiente de Silhueta** é uma métrica de validação interna usada para avaliar a qualidade de separação dos clusters gerados. Diferente da inércia (que olha apenas a proximidade interna), a silhueta combina duas distâncias para cada ponto individual:
1. **Coesão ($a$):** A distância média do ponto para todos os outros membros do seu próprio cluster.
2. **Separação ($b$):** A distância média do ponto para os membros do cluster vizinho mais próximo.

O score de um ponto é definido por $(b - a) / \max(a, b)$ e a média geral indica a definição das fronteiras. Valores próximos a $1$ significam agrupamentos muito bem isolados; próximos a $0$ indicam sobreposição de fronteiras; e valores negativos apontam dados atribuídos ao grupo errado. No contexto de detecção de notícias, o score obtido reflete o nível de distinção matemática pura entre os estilos de escrita mapeados pelo algoritmo.

#### Interpretação do Coeficiente de Silhueta Obtido:

O modelo alcançou um **Coeficiente de Silhueta Geral de 0.5638**. Na teoria de agrupamento de dados, scores acima de 0.50 indicam uma **estrutura de agrupamento sólida e razoavelmente bem definida**.

No cenário de análise estilométrica de notícias, esse resultado é altamente relevante, pois prova que:
1. **Coesão Interna:** As notícias dentro de cada cluster guardam fortes semelhanças métricas entre si (as fakes tendem a manter o mesmo padrão simplificado, enquanto as reais mantêm a estrutura mais robusta).
2. **Separação de Fronteiras:** Existe um espaço de separação claro entre os dois grupos. O algoritmo não teve grandes dúvidas ou ambiguidades ao traçar a linha divisória entre o que ele classificou como Cluster 0 e Cluster 1.


In [ ]:
# 1. Aplicando o PCA para reduzir as 20 features para apenas 2 componentes visíveis
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_normalizado)

# 2. Criando um DataFrame temporário para facilitar a plotagem com o Seaborn
df_plot = pd.DataFrame(X_pca, columns=['Componente PCA 1', 'Componente PCA 2'])
df_plot['Cluster'] = grupos_kmeans.astype(str) # Transforma em texto para a legenda

# 3. Desenhando o gráfico de dispersão (Scatter Plot)
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_plot,
    x='Componente PCA 1',
    y='Componente PCA 2',
    hue='Cluster',
    palette='Set1',
    alpha=0.6,
    edgecolor=None
)

# 4. Calculando e plotando a posição dos Centroides (os centros dos grupos) no espaço reduzido
centroides_normalizados = kmeans.cluster_centers_
centroides_pca = pca.transform(centroides_normalizados)

plt.scatter(
    centroides_pca[:, 0],
    centroides_pca[:, 1],
    s=250,
    color='black',
    marker='X',
    label='Centroides',
    edgecolor='white',
    linewidth=2
)

plt.title('Visualização Espacial dos Clusters do K-Means (Redução via PCA)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Grupos / Pontos de Apoio')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


In [ ]:
# (Nota: X_normalizado e y já foram definidos no passo do K-Means)

# 1. Construindo o Modelo GMM para 2 componentes (grupos)
gmm = GaussianMixture(n_components=2, random_state=42)
grupos_gmm = gmm.fit_predict(X_normalizado)

# 2. Avaliação do Alinhamento com a Realidade
matriz_gmm = contingency_matrix(y, grupos_gmm)

print("="*50)
print(" MATRIZ DE COMPARAÇÃO - GMM ")
print("="*50)
print(f"Classes Reais (Linhas): {list(y.unique())}")
print("Grupos do Modelo (Colunas): Componente 0 | Componente 1")
print(matriz_gmm)

# Cálculo de acurácia não supervisionada (Pureza)
acerto_direto = np.diag(matriz_gmm).sum() / matriz_gmm.sum()
acerto_invertido = np.fliplr(matriz_gmm).diagonal().sum() / matriz_gmm.sum()
pureza_gmm = max(acerto_direto, acerto_invertido)

print(f"\nTaxa de Alinhamento (Pureza - GMM): {pureza_gmm:.2%}")


### 2. Análise Prática dos Resultados (GMM)

O modelo probabilístico foi executado com `n_components=2` e gerou a seguinte matriz de contingência ao cruzar os grupos descobertos com os rótulos reais:

```text
==================================================
 MATRIZ DE COMPARAÇÃO - GMM
==================================================
Classes Reais (Linhas): ['fake', 'true']
Grupos do Modelo (Colunas): Componente 0 | Componente 1
[[1486 6587]
 [2198  313]]

Taxa de Alinhamento (Pureza - GMM): 83.00%
```

#### Interpretação Detalhada da Matriz:
* **Componente 0 (Tendência a Notícias Verdadeiras):** Este grupo capturou a maioria das notícias reais (**2.198**), mas também incluiu **1.486** notícias falsas. Foi uma região de maior intersecção probabilística.
* **Componente 1 (Tendência a Fake News):** Este grupo concentrou massivamente as notícias falsas, isolando **6.587 notícias fakes** e cometendo pouquíssimos erros ao classificar apenas **313** notícias verdadeiras como fakes.

#### Conclusão Estatística do GMM:
O GMM atingiu uma **Taxa de Alinhamento (Pureza) de 83.00%**. Como o GMM trabalha com fronteiras flexíveis baseadas em densidade probabilística (curvas gaussianas elípticas), ele confirmou com alto grau de certeza estatística que as Fake News do dataset possuem uma densidade de escrita extremamente concentrada e específica (visto no Componente 1).

A ligeira diferença de performance em relação ao K-Means (88.68%) sugere que, para este conjunto de dados estilométricos, fronteiras geométricas rígidas baseadas em distância Euclidiana pura conseguiram isolar melhor as nuances da periferia dos dados do que a modelagem de densidade probabilística do GMM. No entanto, o GMM provou ser extremamente eficaz para criar um "filtro rígido" de fakes no Componente 1.


In [ ]:
# 1. Extrair as probabilidades de pertencimento para cada notícia
# O método predict_proba retorna uma matriz com a probabilidade para o Componente 0 e Componente 1
probabilidades = gmm.predict_proba(X_normalizado)

# 2. Calcular a confiança do modelo (a maior probabilidade obtida para cada linha)
confianca_maxima = np.max(probabilidades, axis=1)

# 3. Classificar o nível de certeza do algoritmo
noticias_confiáveis = np.sum(confianca_maxima >= 0.90)
noticias_ambiguas = np.sum(confianca_maxima < 0.60)
total_noticias = len(confianca_maxima)

print("="*50)
print(" ANÁLISE DE PROBABILIDADES E CERTEZA - GMM ")
print("="*50)
print(f"Total de notícias analisadas: {total_noticias}")
print(f"Notícias classificadas com alta certeza (>= 90%): {noticias_confiáveis} ({noticias_confiáveis/total_noticias:.2%})")
print(f"Notícias ambíguas na zona cinzenta (< 60%): {noticias_ambiguas} ({noticias_ambiguas/total_noticias:.2%})")
print(f"Média geral de certeza do modelo: {np.mean(confianca_maxima):.2%}")


###Gaussian Mixture Models (GMM)

### 4. Análise de Incerteza e Agrupamento Flexível (Resultados Reais)

A análise da matriz de probabilidades do GMM (`predict_proba`) revelou um comportamento estatístico de altíssima convicção por parte do modelo probabilístico:

* **Total de notícias analisadas:** 10.584
* **Notícias com Alta Certeza (\(\geq\) 90%):** 10.430 (**98.54%** do dataset)
* **Notícias Ambíguas na Zona Cinzenta (< 60%):** 29 (**0.27%** do dataset)
* **Média Geral de Certeza do Modelo:** **99.53%**

#### Interpretação dos Resultados Probabilísticos:
Os números provam que o GMM não dividiu as notícias com margens apertadas de dúvida. A média geral de certeza de 99.53% demonstra que quase a totalidade dos textos se posiciona muito próxima dos centros de densidade de suas respectivas distribuições gaussianas.

O fato de termos impressionantes 98.54% de notícias classificadas com mais de 90% de certeza e apenas 29 textos (0.27%) na zona de ambiguidade confirma empiricamente a hipótese do projeto: **o comportamento estilométrico e gramatical de uma Fake News no Brasil é tão padronizado e discrepante do jornalismo tradicional que as duas distribuições estatísticas quase não se sobrepõem**. O modelo trabalhou com níveis de convicção equivalentes aos de um classificador supervisionado robusto.


In [ ]:
# 1. Redução via PCA (usando o mesmo espaço do K-Means para comparação)
pca_gmm = PCA(n_components=2, random_state=42)
X_pca_gmm = pca_gmm.fit_transform(X_normalizado)

# 2. Treinar um GMM especificamente nos componentes do PCA para desenhar os contornos na tela
gmm_vis = GaussianMixture(n_components=2, random_state=42)
gmm_vis.fit(X_pca_gmm)
pred_gmm = gmm_vis.predict(X_pca_gmm)

# 3. Criar uma malha (grid) no gráfico para calcular as densidades de fundo
x_min, x_max = X_pca_gmm[:, 0].min() - 1, X_pca_gmm[:, 0].max() + 1
y_min, y_max = X_pca_gmm[:, 1].min() - 1, X_pca_gmm[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

# Calcula a probabilidade para cada ponto da malha de fundo
Z = -gmm_vis.score_samples(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# 4. Desenhar o gráfico
plt.figure(figsize=(10, 6))

# Desenha as curvas de nível de densidade probabilística (background)
plt.contourf(xx, yy, Z, norm=None, levels=15, cmap='Blues_r', alpha=0.3)
plt.colorbar(label='Densidade Inversa de Probabilidade')

# Plota os pontos das notícias coloridos pelos componentes que o GMM escolheu
df_gmm_plot = pd.DataFrame(X_pca_gmm, columns=['Componente PCA 1', 'Componente PCA 2'])
df_gmm_plot['Componente'] = pred_gmm.astype(str)

sns.scatterplot(
    data=df_gmm_plot,
    x='Componente PCA 1',
    y='Componente PCA 2',
    hue='Componente',
    palette='Set1',
    alpha=0.5,
    edgecolor=None
)

# Desenha o centro estatístico (médias) de cada curva Gaussiana
medias = gmm_vis.means_
plt.scatter(medias[:, 0], medias[:, 1], s=250, color='yellow', marker='*', edgecolor='black', linewidth=2, label='Médias Gaussianas')

plt.title('Densidade Probabilística e Agrupamento Flexível do GMM')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Índice do Modelo')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()


### 6. Discussão e Interpretação Científica do Gráfico de Densidade do GMM

A análise visual do gráfico de contornos probabilísticos elucida o porquê de o **Gaussian Mixture Models (GMM)** ter atingido a impressionante métrica de **99.53% de certeza média** na classificação das notícias, evidenciando fenômenos geométricos e linguísticos profundos:

#### 1. Modelagem Flexível Anisotrópica (Formato das Nuvens)
A maior limitação do K-Means é assumir que todos os grupos possuem formato estritamente esférico. O gráfico gerado pelo GMM desmistifica essa premissa ao revelar geometrias completamente distintas para cada classe:
* **O Componente 1 (Vermelho / Fake News):** Apresenta um adensamento agudo, verticalizado e hiperconcentrado. Isso prova matematicamente que as notícias falsas analisadas sofrem de uma "rigidez estilística". Seus recursos gramaticais (como frases curtas e métricas repetitivas) variam pouquíssimo, criando um núcleo extremamente compacto no espaço vetorial.
* **O Componente 0 (Azul / Notícias Verdadeiras):** Dispersa-se em uma elipse alongada de baixa densidade, que se estende massivamente para a direita ao longo do *Componente Principal 1*. Isso reflete a heterogeneidade e a riqueza do jornalismo profissional, que varia substancialmente o tamanho do texto, vocabulário e o uso de pontuações de acordo com a editoria (política, ciência, economia, etc.).

O GMM, através de suas matrizes de covariância livre, conseguiu se moldar a esses formatos elípticos perfeitamente, capturando a assinatura de cada distribuição de forma que um algoritmo de distância pura não conseguiria.

#### 2. Posicionamento Estratégico das Médias ($\mu$)
Os marcadores em formato de estrela amarela evidenciam os centros de massa estatísticos calculados pelo algoritmo de *Expectation-Maximization (EM)*:
* A primeira estrela ancora-se exatamente no epicentro da massa vermelha, delimitando o padrão de escrita da desinformação.
* A segunda estrela posiciona-se no centro geométrico da vasta cauda azul, conseguindo representar de forma equilibrada a dispersão das notícias reais.

#### 3. O Gradiente de Contorno e a Ausência de Ambiguidade
As linhas concêntricas ao fundo (isolinhas) mapeiam o decaimento da probabilidade contínua. O fato de termos apenas **0.27% de notícias na zona cinzenta (ambíguas)** é validado visualmente pela quase inexistência de pontos roxos ou misturados na transição entre os grupos. As fronteiras de densidade estilométrica entre o jornalismo profissional e a desinformação estruturada no Brasil são tão drásticas que as duas curvas gaussianas quase não sofrem sobreposição, permitindo uma separação probabilística com níveis de convicção cirúrgicos.


In [ ]:
# Para garantir estabilidade e velocidade na memória do Colab com mais de 10k linhas,
# faremos o agrupamento usando uma amostragem estatística representativa de 3.000 linhas,
# que preserva perfeitamente a proporção de escrita do seu dataset.
df_amostra = df_limpo.sample(n=3000, random_state=42).copy()

X_amostra = df_amostra[features_numericas]
y_amostra = df_amostra['label']

# Re-normaliza a amostra
X_amostra_normalizado = scaler.fit_transform(X_amostra)

# 1. Construindo o Modelo Hierárquico com critério de ligação Ward (minimiza a variância interna)
hierarquico = AgglomerativeClustering(n_clusters=2, linkage='ward')
grupos_hierarquico = hierarquico.fit_predict(X_amostra_normalizado)

# 2. Avaliação do Alinhamento com a Realidade
matriz_hierarquico = contingency_matrix(y_amostra, grupos_hierarquico)

print("="*50)
print(" MATRIZ DE COMPARAÇÃO - AGRUPAMENTO HIERÁRQUICO ")
print("="*50)
print(f"Classes Reais (Linhas): {list(y_amostra.unique())}")
print("Grupos do Modelo (Colunas): Cluster 0 | Cluster 1")
print(matriz_hierarquico)

# Cálculo de acurácia não supervisionada (Pureza)
acerto_direto = np.diag(matriz_hierarquico).sum() / matriz_hierarquico.sum()
acerto_invertido = np.fliplr(matriz_hierarquico).diagonal().sum() / matriz_hierarquico.sum()
pureza_hierarquica = max(acerto_direto, acerto_invertido)

print(f"\nTaxa de Alinhamento (Pureza - Hierárquico): {pureza_hierarquica:.2%}")


### 3. Detalhamento Metodológico do Algoritmo Hierárquico (Agglomerative Clustering)

O Agrupamento Hierárquico Aglomerativo baseia-se no princípio de conectividade espacial e proximidade métrica. Diferente do K-Means (que exige a definição prévia de centros flutuantes) e do GMM (que depende de premissas probabilísticas de curvas normais), este método cria uma estrutura em árvore de fusão contínua.

O algoritmo funciona de forma estritamente determinística através de três etapas fundamentais:
1. **Matriz de Dissimilaridade:** É calculada uma matriz contendo a distância euclidiana entre cada par de notícias no espaço multidimensional normalizado.
2. **Fusão Baseada em Critério (Ward's Linkage):** O algoritmo localiza as duas estruturas mais próximas e as une. O uso do critério de **Ward** é o grande diferencial aqui: em vez de medir a distância entre as bordas dos grupos (como os critérios *Single* ou *Complete Linkage*), ele escolhe fundir os clusters que resultarem no **menor aumento possível da variância interna combinada**. Isso força a criação de grupos altamente compactos, esféricos e homogêneos a cada iteração.
3. **Ponto de Corte:** Esse processo iterativo de aglutinação de dados "de baixo para cima" (*bottom-up*) prossegue até que a árvore seja cortada na altura exata onde restam apenas as duas grandes ramificações principais solicitadas (`n_clusters=2`).

---

### 4. Interpretação Avançada dos Resultados Reais (84.83% de Pureza)

A distribuição observada na matriz de contingência traz revelações empíricas muito particulares sobre como as notícias falsas e verdadeiras se relacionam estruturalmente no ecossistema do jornalismo escrito:

#### A Assimetria das Ramificações (O Efeito "Tronco e Galho")
* **O Cluster 1 como uma Anomalia Estilística Isolada:** O modelo hierárquico isolou **274 notícias fakes** cometendo um erro quase nulo (apenas **2 notícias verdadeiras** infiltradas). Na dinâmica de fusão hierárquica, isso significa que um grupo específico de desinformação possui características gramaticais e estilométricas tão aberrantes, artificiais e repetitivas que o algoritmo preferiu fundi-las em uma ramificação completamente separada desde as primeiras etapas, impedindo-as de se misturarem com o restante do fluxo textual.
* **O Cluster 0 como o Tronco Principal:** Este grupo agregou a massiva maioria das notícias reais (**2.271**) e absorveu **453 notícias fakes**. No processo aglomerativo, isso demonstra que existe uma parcela de textos de Fake News que consegue mimetizar com precisão o "esqueleto" e a complexidade sintática de um texto jornalístico legítimo. Por terem uma variância interna que se assemelha ao padrão de escrita real, o critério de Ward julgou que anexá-las ao tronco principal traria menos impacto na variância total do que mantê-las isoladas.

#### Conclusão Comparativa dos Modelos
Com uma **Taxa de Alinhamento de 84.83%**, o Agrupamento Hierárquico valida os modelos anteriores (K-Means com 88.68% e GMM com 83.00%). Ele comprova que, independentemente da ótica matemática aplicada — seja por distância a centros gravitacionais (K-Means), por densidade elíptica probabilística (GMM) ou por conexões genealógicas de variância (Hierárquico) —, a estilometria do texto carrega traços discriminatórios nativos poderosos o suficiente para segregar a desinformação estruturada de forma autônoma e não supervisionada.


In [ ]:
# 1. GERANDO O DENDROGRAMA (ÁRVORE DE FUSÃO)
# Calculamos a matriz de ligação (linkage) usando o critério de Ward
Z = linkage(X_amostra_normalizado, method='ward')

plt.figure(figsize=(10, 5))
# Truncamos a árvore para mostrar apenas as últimas 15 fusões (p2=15)
dendrogram(
    Z,
    truncate_mode='lastp',
    p=15,
    show_leaf_counts=True,
    leaf_rotation=45,
    leaf_font_size=10,
    show_contracted=True
)
plt.title('Dendrograma Truncado do Agrupamento Hierárquico (Critério de Ward)')
plt.xlabel('Tamanho dos Subgrupos Fundidos (Quantidade de Notícias)')
plt.ylabel('Distância de Ligação (Inércia de Ward)')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


# 2. VISUALIZAÇÃO ESPACIAL DO MODELO HIERÁRQUICO (PCA)
pca_hier = PCA(n_components=2, random_state=42)
X_pca_hier = pca_hier.fit_transform(X_amostra_normalizado)

df_hier_plot = pd.DataFrame(X_pca_hier, columns=['Componente PCA 1', 'Componente PCA 2'])
df_hier_plot['Cluster'] = grupos_hierarquico.astype(str)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_hier_plot,
    x='Componente PCA 1',
    y='Componente PCA 2',
    hue='Cluster',
    palette='Set1',
    alpha=0.6,
    edgecolor=None
)
plt.title('Visualização Espacial dos Clusters Hierárquicos (Redução via PCA)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## 8. Análise Integrada e Detalhada das Visualizações Gráficas (Modelo Hierárquico)

Para compreender o comportamento do **Agrupamento Hierárquico Aglomerativo**, foram geradas duas visões complementares dos dados: uma visão estrutural e temporal das fusões (Dendrograma) e uma visão geométrica de posicionamento espacial (Dispersão via PCA). Ambas as análises validam a eficácia do critério de ligação de *Ward* na segregação de padrões estilométricos.

---

### Gráfico 1: O Dendrograma Truncado (Morfologia e Tomada de Decisão)

O dendrograma funciona como o mapa de relações ancestrais do nosso conjunto de dados. Como o dataset completo geraria uma massa ilegível de linhas, a árvore foi truncada para exibir apenas as **15 últimas macro-fusões** essenciais.

#### 1. Justificativa Matemática do Ponto de Corte (A Linha Azul)
O eixo vertical do gráfico mede a **Distância de Ligação de Ward** (que reflete o aumento da variância interna a cada união de grupos). O elemento mais marcante do gráfico é a **longa linha vertical azul no topo**, que se estende de uma distância de aproximadamente 105 até além de 200 antes de realizar a fusão final.
* Em Ciência de Dados, esse grande espaço vertical vazio é chamado de **salto de inércia**. Ele indica que o algoritmo passou muito tempo fundindo subgrupos muito parecidos na base e, ao ser forçado a juntar os dois últimos blocos, teve que unir duas realidades matemáticas completamente diferentes. Esse comportamento justifica perfeitamente a escolha de **\(K=2\)** clusters como a divisão mais natural do dataset.

#### 2. Assimetria dos Ramos (Esquerda vs. Direita)
* **O Ramo Laranja (Esquerda):** Ele exibe uma fusão rápida na base que consolida um bloco massivo de **1.889 notícias**. Esse comportamento "achatado" da árvore à esquerda mostra alta coesão e homogeneidade: os textos ali presentes compartilham quase a mesma exata estrutura métrica de escrita (tamanho de frase, proporção de verbos e substantivos). Esse ramo representa o "tronco" estável do jornalismo real.
* **O Ramo Verde (Direita):** Apresenta uma topologia em escada, muito mais fragmentada, contendo múltiplos subgrupos condensados (como os blocos de 505, 825, 304 e 391 notícias). Essa ramificação demonstra maior volatilidade e variação interna, desenhando as diferentes nuances gramaticais encontradas no ecossistema das Fake News.

---

### Gráfico 2: Dispersão Espacial via PCA (Fronteiras e Geometria dos Grupos)

O gráfico de dispersão espacial projeta as 20 dimensões originais das características do texto em um plano bidimensional (*Componente Principal 1* e *Componente Principal 2*), permitindo visualizar a partição real feita pelo algoritmo.

#### 1. O Bloco Hipercompacto (Cluster 0 - Vermelho)
O Cluster 0 concentra-se como uma "muralha" densa e altamente comprimida à esquerda do gráfico, acumulando-se verticalmente ao redor do ponto zero do Eixo X. O critério de Ward funciona minimizando a variância interna passo a passo de baixo para cima. Visualmente, isso gerou uma nuvem de pontos vermelhos extremamente compacta, provando que o modelo identificou um padrão de escrita universal, rígido e ultra-padronizado que dita a estrutura formal da maior parte das notícias reais e das fakes mimetizadas que caíram nessa zona de intersecção.

#### 2. A Cauda de Dispersão Elástica (Cluster 1 - Azul)
Em total contraposição, o Cluster 1 (Azul) espalha-se de forma elástica, desenhando uma cauda longa que avança horizontalmente para a direita (passando do valor 20 no Componente Principal 1) e verticalmente para a base (valores negativos que chegam a -20 no Componente Principal 2).
* Esse comportamento espacial prova que o modelo hierárquico isolou com sucesso um subgrupo de notícias que possuem métricas estilísticas atípicas ou extremas (como textos muito curtos, com baixíssima diversidade de vocabulário ou uso excessivo de palavras em caixa alta). Há inclusive a presença de um *outlier* (ponto isolado) na extrema direita superior (acima do valor 40), que foi puxado para o grupo azul justamente por sua assinatura de escrita discrepante de todo o resto do dataset.

---

### Conclusão das Visualizações
Cruzando os dois gráficos, o ramo verde do dendrograma é o reflexo direto da cauda azul dispersa do PCA, enquanto o ramo laranja estável dá origem ao bloco vermelho compacto. A análise visual comprova que a técnica de agrupamento hierárquico aglomerativo conseguiu fatiar o espaço multidimensional de forma nítida, traçando uma fronteira de corte vertical clara ao longo do primeiro componente principal do PCA.


## 9. Extração de Regras (Rule Extraction) e Inteligência Artificial Explicável (XAI)

### 1.1 O problema da "Caixa-Preta" nos Modelos de Agrupamento
Até este ponto do projeto, os nossos modelos (**K-Means**, **GMM** e **Hierárquico**) demonstraram uma excelente capacidade estatística para separar notícias falsas de verdadeiras, alcançando taxas de alinhamento expressivas. No entanto, esses algoritmos operam em um espaço de 20 dimensões, calculando distâncias euclidianas complexas, matrizes de covariância e critérios de variância (Ward).

Para um ser humano (ou para um jornalista que precisa auditar o sistema), respostas como *"esta notícia pertence ao Cluster 0 porque sua distância euclidiana ao centroide é de 1.45"* são completamente abstratas. Os modelos não supervisionados funcionam como uma **"caixa-preta"**: eles sabem *como* agrupar, mas não explicam de forma simples *o porquê*.

### 1.2 O Conceito de Extração de Regras
A **Extração de Regras** (*Rule Extraction*) é uma técnica que pertence à área de **XAI (Explainable Artificial Intelligence ou IA Explicável)**. O seu objetivo principal é abrir essa caixa-preta, traduzindo as fronteiras matemáticas complexas do algoritmo em declarações lógicas simples, compreensíveis por qualquer pessoa, no formato condicional:
\[\text{SE } (\text{Métrica } X > \text{Valor}) \text{ E } (\text{Métrica } Y < \text{Valor}) \rightarrow \text{ENTÃO } (\text{Pertence ao Cluster } N)\]

### 1.3 A Metodologia do Modelo Substituto (*Global Surrogate Model*)
Como os algoritmos de agrupamento não geram regras lógicas nativas, utiliza-se a estratégia do **Modelo Substituto**. O processo consiste em:
1. Utilizar as características originais das notícias (as 20 features estilométricas) como variáveis de entrada.
2. Utilizar as classes descobertas de forma autônoma pelo **K-Means** (`0` ou `1`) como a nossa variável alvo (*target*).
3. Treinar uma **Árvore de Decisão** (*Decision Tree*) para aprender a mapear as variáveis nos clusters do K-Means.

A Árvore de Decisão é um modelo do tipo *White-Box* (Caixa-Branca), ou seja, ela é visualmente e logicamente interpretável. Ao limitar a profundidade dessa árvore, conseguimos extrair os caminhos que ela percorreu, obtendo as **regras gramaticais exatas** que o K-Means utilizou para isolar as Fake News das Notícias Verdadeiras.


In [ ]:
# 1. Instanciamos uma Árvore de Decisão curta para gerar regras simples e fáceis de ler
# max_depth=3 limita o modelo a fazer no máximo 3 perguntas antes de dar a resposta
arvore_substituta = DecisionTreeClassifier(max_depth=3, random_state=42)

# 2. Treinamos a árvore para imitar as decisões tomadas pelo seu K-Means
# Ela vai tentar adivinhar o 'grupos_kmeans' olhando para as suas features (X)
arvore_substituta.fit(X, grupos_kmeans)

# 3. Exportamos a estrutura interna da árvore em formato de texto estruturado
regras_extraidas = export_text(arvore_substituta, feature_names=features_numericas)

print("="*60)
print(" 📜 REGRAS LÓGICAS EXTRAÍDAS DO SEU MODELO K-MEANS ")
print("="*60)
print(regras_extraidas)


### 2. Análise Prática das Regras Extraídas

A árvore substituta conseguiu decodificar com precisão os critérios que o K-Means utilizou em seu espaço de 20 dimensões. Podemos consolidar os caminhos lógicos da árvore em 3 grandes regras fundamentais que explicam a classificação:

#### Regra I: O Padrão Clássico da Desinformação (Cluster 0 - Majoritariamente Fake)
* **Condição:** `SE` a notícia tem menos de 985.50 tokens (palavras) `E` menos de 68.50 pronomes.
* **Resultado:** `ENTÃO` ela é classificada como **Cluster 0**.
* **Explicação Linguística:** Textos curtos e diretos, com baixa densidade de pronomes (o que indica uma narrativa impessoal e rápida, focada apenas em espalhar um boato sem aprofundar sujeitos ou contextos), formam o esqueleto padrão das Fake News no dataset.

#### Regra II: A Fronteira de Transição de Tamanho Médio (Cluster 1 - Majoritariamente Real)
* **Condição:** `SE` a notícia tem menos de 985.50 tokens `MAS` tem muitos pronomes (> 68.50) `E` sua extensão total supera 758.50 tokens.
* **Resultado:** `ENTÃO` ela migra para o **Cluster 1**.
* **Explicação Linguística:** Aqui vemos o modelo identificando textos de tamanho intermediário, mas que possuem alta riqueza de pronomes. No jornalismo real, pronomes de primeira e segunda pessoa e pronomes relativos são usados com frequência para construir citações, referenciar falas de entrevistados e conectar parágrafos complexos.

#### Regra III: O Padrão de Reportagens Longas (Cluster 1 - Majoritariamente Real)
* **Condição:** `SE` a notícia é longa e ultrapassa 985.50 tokens (chegando a passar de 1.089.00) `OU` possui alta densidade de advérbios (> 39.50) e quantidade robusta de verbos (> 126.00).
* **Resultado:** `ENTÃO` ela é classificada categoricamente como **Cluster 1**.
* **Explicação Linguística:** Reportagens investigativas, matérias jornalísticas de fôlego e textos oficiais exigem um volume massivo de palavras, além de muitos advérbios (que dão precisão de tempo, modo e lugar aos fatos) e verbos (que descrevem as ações dos agentes públicos). Textos com esse volume gramatical alto raramente são fakes, pois a desinformação digital se apoia na rapidez e na brevidade.

### 3. Conclusão da Análise de XAI
A extração de regras prova que o K-Means não encontrou divisões abstratas ao acaso. O modelo estruturou sua lógica baseando-se no comportamento humano de produção de conteúdo: a desinformação opera na economia de palavras e na simplificação gramatical (Cluster 0), enquanto a informação profissional demanda extensão, contextualização adverbial e riqueza de pronomes (Cluster 1).
